# 02 - Data Cleaning

## Objective

The goal of this notebook is to clean the IBM Telco Customer Churn dataset and prepare a reliable dataset for exploratory data analysis and machine learning.

The main cleaning tasks are:

- Check duplicate records
- Investigate missing or blank values
- Correct incorrect data types
- Clean categorical values
- Validate numerical values
- Prepare a binary churn indicator for analysis

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Shape:", df.shape)

Shape: (7043, 21)


## Initial Check

Before performing any modifications, the dataset is loaded again from the raw CSV file.

Keeping each notebook independent makes the project easier to reproduce and prevents one notebook from depending on variables created in another notebook.

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df["customerID"].duplicated().sum()

np.int64(0)

## Duplicate Records

No duplicate rows were found in the dataset.

The `customerID` column was also checked separately because every row should represent one unique customer.

In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Missing Values

The standard missing-value check does not report any null values.

However, some missing information may be represented using empty strings rather than `NaN`, so important columns must also be inspected manually.

In [7]:
df["TotalCharges"].dtype

<StringDtype(storage='python', na_value=nan)>

In [8]:
df["TotalCharges"].unique()[:20]

<StringArray>
[  '29.85',  '1889.5',  '108.15', '1840.75',  '151.65',   '820.5',  '1949.4',
   '301.9', '3046.05', '3487.95',  '587.45',   '326.8',  '5681.1',  '5036.3',
 '2686.05', '7895.15', '1022.95', '7382.25',  '528.35',  '1862.9']
Length: 20, dtype: str

In [9]:
blank_total_charges = df["TotalCharges"].str.strip() == ""

blank_total_charges.sum()

np.int64(11)

In [10]:
df.loc[
    blank_total_charges,
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


## Cleaning `TotalCharges`

`TotalCharges` should be numerical, but it is stored as a string.

Further inspection shows that 11 records contain blank values instead of actual missing values.

All of these customers have a tenure of 0 months. This suggests that they are new customers who have not yet accumulated total charges.

In [11]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [12]:
df["TotalCharges"].isnull().sum()

np.int64(11)

In [13]:
df.loc[
    df["TotalCharges"].isnull(),
    ["tenure", "MonthlyCharges", "TotalCharges"]
]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


In [14]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [15]:
df["TotalCharges"].isnull().sum()

np.int64(0)

In [16]:
df["TotalCharges"].dtype

dtype('float64')

### Cleaning Decision

The blank values were first converted to `NaN` using `pd.to_numeric(errors="coerce")`.

Because all affected customers have a tenure of 0 months, their `TotalCharges` values were replaced with 0 rather than using a mean or median imputation.

This preserves the business meaning of the data.

In [17]:
object_cols = df.select_dtypes(include="object").columns

for col in object_cols:
    df[col] = df[col].str.strip()

C:\Users\gogoi\AppData\Local\Temp\ipykernel_18392\1838545402.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns


## Cleaning Categorical Values

Leading and trailing whitespace was removed from categorical columns.

This prevents values such as `"Yes"` and `"Yes "` from being interpreted as different categories.

In [18]:
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}:")
    print(df[col].unique())


customerID:
<StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str

gender:
<StringArray>
['Female', 'Male']
Length: 2, dtype: str

Partner:
<StringArray>
['Yes', 'No']
Length: 2, dtype: str

Dependents:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

PhoneService:
<StringArray>
['No', 'Yes']
Length: 2, dtype: str

MultipleLines:
<StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str

InternetService:
<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

OnlineSecurity:
<StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str

OnlineBackup:
<StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str

DeviceProtection:
<StringArray>
['No', 'Yes', 'No internet 

C:\Users\gogoi\AppData\Local\Temp\ipykernel_18392\358977919.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


## Category Validation

The categorical variables were inspected for inconsistent or unexpected values.

Values such as `No internet service` and `No phone service` were retained because they contain meaningful information and are different from simply answering `No`.

In [19]:
df[["tenure", "MonthlyCharges", "TotalCharges"]].describe()

,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761692,2279.734304
std,24.559481,30.090047,2266.794470
min,0.000000,18.250000,0.000000
25%,9.000000,35.500000,398.550000
50%,29.000000,70.350000,1394.550000
75%,55.000000,89.850000,3786.600000
max,72.000000,118.750000,8684.800000


In [20]:
print("Negative tenure:", (df["tenure"] < 0).sum())
print("Negative MonthlyCharges:", (df["MonthlyCharges"] < 0).sum())
print("Negative TotalCharges:", (df["TotalCharges"] < 0).sum())

Negative tenure: 0
Negative MonthlyCharges: 0
Negative TotalCharges: 0


## Numerical Validation

The numerical variables were checked for impossible values such as negative tenure or negative charges.

No invalid negative values were found.

In [21]:
df["ChurnFlag"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [22]:
df[["Churn", "ChurnFlag"]].head()

,Churn,ChurnFlag
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


## Churn Indicator

The original `Churn` column is kept because its `Yes` and `No` values are easier to interpret during EDA.

A new binary column called `ChurnFlag` is created:

- `0` = Customer did not churn
- `1` = Customer churned

This makes calculations such as churn rate easier.

In [23]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())
print("Unique customers:", df["customerID"].nunique())

df.info()

Rows: 7043
Columns: 22
Duplicate rows: 0
Missing values: 0
Unique customers: 7043
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          

In [24]:
df.to_csv(
    "../data/processed/telco_churn_cleaned.csv",
    index=False
)

## Saving the Cleaned Dataset

The cleaned dataset is saved separately from the original raw dataset.

The raw data is never modified, which makes the cleaning process reproducible and allows us to return to the original source if needed.